In [55]:
import pandas as pd
import numpy as np
import gseapy as gp
import re
from typing import List, Dict, Any
from statsmodels.sandbox.stats.multicomp import multipletests
from scipy.stats import spearmanr
import mygene

In [57]:
mg = mygene.MyGeneInfo()

In [59]:
try:
    exp = pd.read_csv(
        "../GBM.transcriptome__ht_hg_u133a__broad_mit_edu__Level_3__gene_rma__data.data.txt",
        sep='\t',
        skiprows=[1],
        index_col=0
    )
except FileNotFoundError:
    print("Error: Expression file not found. Please ensure 'GBM.transcriptome__ht_hg_u133a__broad_mit_edu__Level_3__gene_rma__data.data.txt' is in the directory.")
    exit()

In [60]:
try:
    d_k = pd.read_csv("../Dks_7_30_25.csv")
except FileNotFoundError:
    print("Error:Dks_7_30_25.csv is not in the directory.")
    exit()

In [61]:
d_k.set_index("id", inplace=True)

In [65]:
exp.columns = [col[:12] for col in exp.columns]

In [67]:
inc_samples = list(set(exp.columns) & set(d_k.index))

In [69]:
exp = exp[inc_samples]
d_k = d_k.loc[inc_samples]

In [71]:
raf_ids = d_k.index.tolist()

In [73]:
exp.index = [gene.upper() for gene in exp.index]

In [75]:
exp = exp.apply(pd.to_numeric, axis=0, errors='coerce').fillna(0)

In [77]:
libs = gp.get_library_name(organism='Human')


In [78]:
gene_sets_dict = gp.get_library(
    name='Reactome_Pathways_2024',
    organism='Human'
)
# Ensure all names are uppercase for case-insensitive matching later
for pathway in gene_sets_dict:
    gene_sets_dict[pathway] = [g.upper() for g in gene_sets_dict[pathway]]
#alias mapping 
# 1. Collect all unique gene names from both sources
exp_genes = set(exp.index)
gs_genes = set(g for sublist in gene_sets_dict.values() for g in sublist)
all_unique_names = list(exp_genes.union(gs_genes))

print(f"Total unique identifiers across EXP and Gene Sets: {len(all_unique_names)}")
# 2. Query MyGene.info once for all unique names
mapping_results = mg.querymany(
    all_unique_names,
    scopes='symbol,alias',  # Search using both official symbol and known aliases
    fields='symbol',
    species='human',
    as_dataframe=True
)

# 3. Create a reliable {OLD_NAME: NEW_SYMBOL} map
gene_rename_map = {}
for old_name, row in mapping_results.iterrows():
    # Only map if a high-confidence official symbol was found
    if row.get('_score', 0) > 0 and pd.notna(row.get('symbol')):
        new_symbol = row['symbol'].upper()
        gene_rename_map[old_name] = new_symbol

print(f"Created a mapping for {len(gene_rename_map)} identifiers.")

# A. Standardize Expression Matrix (exp)
original_exp_rows = exp.shape[0]

# Apply the map, keeping the original name if no standard symbol was found
exp.index = exp.index.map(lambda x: gene_rename_map.get(x, x))

# Aggregate duplicate symbols (resulting from multiple aliases mapping to one symbol) by mean
exp = exp.groupby(level=0).mean()

print(f"EXP matrix standardized. Original rows: {original_exp_rows}, New unique genes: {exp.shape[0]}")


# B. Standardize Gene Sets (gene_sets_dict)
for pathway, genes in gene_sets_dict.items():
    updated_genes = []
    for gene in genes:
        updated_genes.append(gene_rename_map.get(gene, gene))

    exp_standardized_genes = set(exp.index)

    final_genes = []
    for g in set(updated_genes): # Use set to remove duplicates
        if g in exp_standardized_genes:
            final_genes.append(g)

    gene_sets_dict[pathway] = final_genes


INFO:biothings.client:querying 1-1000 ...


Total unique identifiers across EXP and Gene Sets: 16283


INFO:biothings.client:querying 1001-2000 ...
INFO:biothings.client:querying 2001-3000 ...
INFO:biothings.client:querying 3001-4000 ...
INFO:biothings.client:querying 4001-5000 ...
INFO:biothings.client:querying 5001-6000 ...
INFO:biothings.client:querying 6001-7000 ...
INFO:biothings.client:querying 7001-8000 ...
INFO:biothings.client:querying 8001-9000 ...
INFO:biothings.client:querying 9001-10000 ...
INFO:biothings.client:querying 10001-11000 ...
INFO:biothings.client:querying 11001-12000 ...
INFO:biothings.client:querying 12001-13000 ...
INFO:biothings.client:querying 13001-14000 ...
INFO:biothings.client:querying 14001-15000 ...
INFO:biothings.client:querying 15001-16000 ...
INFO:biothings.client:querying 16001-16283 ...
INFO:biothings.client:Finished.
INFO:biothings.client:Pass "returnall=True" to return complete lists of duplicate or missing query terms.


Created a mapping for 15886 identifiers.
EXP matrix standardized. Original rows: 12042, New unique genes: 11995


In [79]:
gsva_results = gp.gsva(
    data=exp,
    gene_sets=gene_sets_dict,
    min_size=5,
    method='gsva',
    outdir=None,
    no_plot=True,
    processes=1
)

In [81]:
gsva_output=gsva_results.res2d.pivot(index='Term', columns='Name', values='ES')

In [82]:
gsva_output.to_csv("reactome_path_gsva_scores_11_19_2026.csv")

In [83]:
merged=pd.merge( d_k, gsva_output.T, left_on='id',  right_on='Name',  how='inner')

In [84]:
tmp=merged.iloc[:,3:]

In [85]:
pathways=tmp.columns
results_list=[]
for i in range(tmp.shape[1]):
    pat=pd.to_numeric(tmp.iloc[:, i], errors="coerce").to_numpy(dtype=float)
    coefficient, p_value = spearmanr(np.array(merged["D/k"]),pat)
    path_name=pathways[i]
    results_list.append({
        'Pathway': path_name,
        'Spearman_Rho': coefficient,
        'P_Value': p_value})
correlation_results_df=pd.DataFrame(results_list)

In [86]:
reject, pvals_corrected, _, _ = multipletests(
    correlation_results_df['P_Value'],
    method='fdr_bh'
)

In [87]:
correlation_results_df['FDR_P_Value'] = pvals_corrected

In [88]:
correlation_results_df_sorted = correlation_results_df.sort_values(
    by='Spearman_Rho',
    ascending=False
).reset_index(drop=True)

In [89]:
print("--- Strongest Positive Correlations (Head of DataFrame) ---")
print(correlation_results_df_sorted[["Pathway", "Spearman_Rho" ,  "P_Value" ]].head(10))

--- Strongest Positive Correlations (Head of DataFrame) ---
                                             Pathway  Spearman_Rho   P_Value
0                                 HS-GAG Degradation      0.317837  0.002140
1                Diseases of Carbohydrate Metabolism      0.301083  0.003731
2                             Diseases of Metabolism      0.298168  0.004097
3                          Diseases of Glycosylation      0.274343  0.008499
4                  Defective EXT2 Causes Exostoses 2      0.272591  0.008946
5  Defective EXT1 Causes Exostoses 1, TRPS2 and CHDS      0.272591  0.008946
6                                HS-GAG Biosynthesis      0.260599  0.012601
7                              Mucopolysaccharidoses      0.255184  0.014638
8                        Keratan Sulfate Degradation      0.251473  0.016194
9        Heparan Sulfate Heparin (HS-GAG) Metabolism      0.249275  0.017181


In [90]:
correlation_results_df_sorted.to_csv(
    "ReactomePathway_D_k_correlation_11_19_2026.csv",  float_format="%.4f", index=False
)